# Gemma-3-4B translation inference (via vLLM)

Runs `google/gemma-3-4b-it` through a local vLLM OpenAI-compatible server. Gemma has no
reasoning/thinking mode. Start the server first:

```bash
vllm serve google/gemma-3-4b-it --port 8000
```

Gemma's chat template has no `system` role, so the instruction is folded into the user turn.

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

In [ ]:
ground_truth_translations = pd.read_csv("../../data/english_to_urdu_translations/context_translations_trimmed.csv")

In [ ]:
def get_urdu_response(context_eng):
    client = OpenAI(api_key="EMPTY", base_url="http://localhost:8000/v1")

    instruction = "You are an expert translator specializing in English to Urdu legal translations."
    prompt = f"""Translate the following legal text from English to Urdu. Provide only the translated text in Urdu script, without any additional commentary or explanation.

English Text:
```{context_eng}```

Urdu Translation:"""

    completion = client.chat.completions.create(
        model="google/gemma-3-4b-it",
        temperature=0.0,
        max_tokens=1024,
        messages=[
            {"role": "user", "content": f"{instruction}\n\n{prompt}"},
        ],
    )

    return completion.choices[0].message.content

In [ ]:
from tqdm import tqdm

predicted_translations = []

for i in tqdm(range(len(ground_truth_translations)), desc="Translating"):
    context_eng = ground_truth_translations.iloc[i].context_eng
    predicted_answer = get_urdu_response(context_eng=context_eng)
    predicted_translations.append(predicted_answer)

In [ ]:
predicted_translations[:5]

In [ ]:
ground_truth_translations["predicted_translation_gemma"] = predicted_translations

In [ ]:
ground_truth_translations.to_csv("../../data/model_translation_responses/context_translations_gemma.csv", index=False)